# Notebook 44 · Is the edge alpha or volatility exposure?

**Development-only · three chronological folds · zero transaction costs · closed final holdout.**

This notebook tests Ridge α=100 against *date-matched, factor-matched randomized portfolios*, within-factor directional tests, factor-balanced portfolios, influential-symbol removal, and moving-block confidence intervals. It deliberately prioritizes falsification over finding another highest-return model.

**Interpretation:** Matched portfolios are observational controls, not causal identification. The matching procedure excludes selected stocks when possible, so it measures selection relative to available alternatives. Multiple tests are exploratory. Daily returns are equal-weight arithmetic next-bar returns, not an executable P&L ledger.

## Experiments 1–4 · Load the identical frozen development universe and Ridge pipeline

In [ ]:
from pathlib import Path
import sys,time
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks':ROOT=ROOT.parent
assert (ROOT/'src').is_dir(),f'Run from repository root or notebooks/, not {ROOT}'
sys.path.insert(0,str(ROOT/'src'))
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from IPython.display import display
from stock_ml_dynamic.pooled_models import load_master
from stock_ml_dynamic.classifier_benchmark import eligible_universe
from stock_ml_dynamic.feature_ablation import development_holdout_split
from stock_ml_dynamic.historical_market import add_historical_market_features
from stock_ml_dynamic.accuracy_features import add_accuracy_features
from stock_ml_dynamic.volume_regime_research import add_volume_regime_features
from stock_ml_dynamic.edge_robustness_41 import fit_and_score,run_portfolio_stress
from stock_ml_dynamic.factor_matched_44 import *
OUT=ROOT/'artifacts'/'factor_matched'/'1d';OUT.mkdir(parents=True,exist_ok=True)
master,latest,bundle,manifest=load_master(ROOT/'artifacts/master_tables/1d')
eligible,eligibility=eligible_universe(master)
development,cutoff=development_holdout_split(eligible,.20)
expanded,historical=add_historical_market_features(development)
expanded,technical=add_accuracy_features(expanded)
expanded,features=add_volume_regime_features(expanded)
print('Development',expanded.shape,'cutoff',cutoff,'feature extras',len(features))
assert len(expanded)>0 and len(features)>0

## Experiments 5–8 · Frozen Ridge reconstruction and baseline audit

In [ ]:
t0=time.perf_counter()
scored,coefficients,audit=fit_and_score(expanded,features,cap=60000,seed=42)
base=run_portfolio_stress(scored)
base_metrics=base['metrics'].loc[(base['metrics'].model=='ridge_all')&(base['metrics']['k'].eq(5))].sort_values('fold')
display(audit);display(base_metrics)
print('Ridge fitting elapsed',round(time.perf_counter()-t0,1),'seconds')
assert len(base_metrics)==3 and scored.fold.nunique()==3
assert scored[['fold','bar_timestamp','symbol']].duplicated().sum()==0
print('Available observable factors:',[c for c in FACTORS if c in scored.columns])

## Experiments 9–18 · Date-matched, factor-matched Monte Carlo controls

For each selected top-five stock, choose an alternative stock from the same date with the nearest **observable** factor-bin profile. Match volatility, relative volume, momentum, and volume when present. Run 300 randomized portfolios per day; evaluate each fold's distribution of average returns. No realized returns enter the matching procedure. **The empirical Monte Carlo p-values are not corrected for model search.**

In [ ]:
t0=time.perf_counter()
matched_daily,matched_summary,matched_null,available=matched_random_experiment(scored,k=5,reps=300,seed=42)
display(matched_summary.round(6))
print('Factors:',available,'elapsed',round(time.perf_counter()-t0,1),'seconds')
print('Match distance 0 = all matching bins exact; nonzero = fallback nearest bins')
assert len(matched_summary)==3

## Experiments 19–24 · Matched controls at 10 and 20 stocks per day

In [ ]:
matched_other=[]
for k in (10,20):
    d,s,_,_=matched_random_experiment(scored,k=k,reps=150,seed=42+k)
    matched_other.append(s)
matched_all=pd.concat([matched_summary,*matched_other],ignore_index=True)
display(matched_all.sort_values(['k','fold']).round(6))

## Experiments 25–31 · Within-volatility and within-volume directional discrimination

In [ ]:
directionality=within_bin_directionality(scored)
display(directionality.sort_values(['factor','fold','bin']).round(5))
spread=directionality.groupby(['factor','bin'],as_index=False).agg(mean_spread=('directional_spread','mean'),mean_ic=('ic','mean'),up5=('top_up5','mean'),down5=('top_down5','mean'))
display(spread.round(5))
fig,ax=plt.subplots(figsize=(9,4))
for fac,g in spread.groupby('factor'):
    ax.plot(g['bin'],100*g['mean_spread'],marker='o',label=fac)
ax.axhline(0,color='gray',linewidth=1);ax.set(xlabel='Within-date factor bin (low to high)',ylabel='Top-minus-bottom score return, pp',title='Directional discrimination after factor stratification')
ax.legend();plt.show()

## Experiments 32–37 · Volatility-balanced top-five selection

Allocate one stock to each volatility tercile, then fill remaining slots by the original Ridge score. This is a **fixed, ex-ante** exposure-control test, not an optimized threshold.

In [ ]:
factor='volatility_20' if 'volatility_20' in scored else available[0]
balanced=stratified_topk(scored,k=5,factor=factor)
balanced_metrics,balanced_daily=portfolio_summary(balanced,scored,k=5)
display(balanced_metrics.round(6))
original_daily=base['daily'].loc[(base['daily'].model=='ridge_all')&(base['daily']['k'].eq(5)),['fold','bar_timestamp','daily_return']].rename(columns={'daily_return':'ridge'})
balanced_compare=original_daily.merge(balanced_daily,on=['fold','bar_timestamp'],validate='one_to_one')
balanced_compare['balanced_lift']=balanced_compare.balanced-balanced_compare.ridge
balanced_ci=block_inference(balanced_compare,columns=('balanced_lift',),reps=1200)
display(balanced_ci.round(6))

## Experiments 38–43 · Symbol exclusions and concentration robustness

In [ ]:
exclusions=leave_symbol_out(scored)
display(exclusions.pivot(index='fold',columns='model',values='mean_daily').round(6))
selected=base['trades'].loc[(base['trades'].model=='ridge_all')&(base['trades']['k'].eq(5))]
contributors=selected.groupby('symbol').agg(n=('next_return','size'),sum_return=('next_return','sum'),avg_return=('next_return','mean')).sort_values('sum_return',ascending=False).reset_index()
display(contributors.head(20).round(5))
for fold,g in selected.groupby('fold'):
    n=max(1,int(np.ceil(.01*len(g))))
    print('Fold',fold,'top 1% winners / net sum:',round(float(g.next_return.nlargest(n).sum()/g.next_return.sum()),3))

## Experiments 44–50 · Date-block bootstrap for matched alpha and market excess

In [ ]:
matched_ci=block_inference(matched_daily,reps=1200,seed=42)
display(matched_ci.round(6))
print('Interpretation: CIs crossing zero do not establish a positive selection advantage.')
fig,ax=plt.subplots(figsize=(9,4))
for fold,g in matched_daily.groupby('fold'):
    ax.plot(pd.to_datetime(g.bar_timestamp),g.matched_excess.rolling(40,min_periods=20).mean()*100,label=f'Fold {fold}')
ax.axhline(0,color='gray',linewidth=1);ax.set(ylabel='40-day rolling matched excess, pp',title='Time stability of matched-stock alpha')
ax.legend();plt.show()

## Experiments 51–56 · Factor balance and placebo diagnostics

In [ ]:
factor_frame,available=add_factor_bins(scored)
selected_keys=selected[['fold','bar_timestamp','symbol']].drop_duplicates()
selected_factors=factor_frame.merge(selected_keys,on=['fold','bar_timestamp','symbol'],how='inner',validate='one_to_one')
balance=[]
for fac in available:
    col='bin_'+fac
    for b in range(3):
        balance.append(dict(factor=fac,bin=b,selected_share=float((selected_factors[col]==b).mean()),universe_share=float((factor_frame[col]==b).mean())))
balance=pd.DataFrame(balance)
display(balance.round(4))
# Negative control: strictly monotone score transformation must not change selections.
original=rank_with_exclusions(scored,scored.score.to_numpy(),5)
monotone=rank_with_exclusions(scored,np.arcsinh(scored.score.to_numpy()),5)
assert original[['bar_timestamp','symbol']].reset_index(drop=True).equals(monotone[['bar_timestamp','symbol']].reset_index(drop=True))
print('PASS: monotone ranking invariance')

## Experiments 57–62 · Decision table, audit and exports

**Interpretation rule:** The Ridge signal is more credible if matched excess is positive in all three folds, its date-block confidence intervals are predominantly above zero, and within-factor directional spreads remain positive. No individual finding establishes deployability. Monte Carlo and bootstrap tests are exploratory and not multiplicity-adjusted.

In [ ]:
decision=matched_summary[['fold','observed','matched_random_mean','excess','monte_carlo_p','mean_bin_distance']].merge(matched_ci.loc[matched_ci.comparison=='matched_excess',['fold','ci_low','ci_high']],on='fold',validate='one_to_one')
decision['positive_matched_excess']=decision.excess>0
decision['block_ci_above_zero']=decision.ci_low>0
display(decision.round(6))
print('Positive matched excess folds:',int(decision.positive_matched_excess.sum()),'/3')
print('Block CI entirely above zero folds:',int(decision.block_ci_above_zero.sum()),'/3')
save_tables(OUT,baseline=base_metrics,training_audit=audit,matched_daily=matched_daily,matched_summary=matched_summary,matched_all_sizes=matched_all,matched_null=matched_null,directionality=directionality,directional_spread=spread,balanced_metrics=balanced_metrics,balanced_ci=balanced_ci,exclusions=exclusions,contributors=contributors,matched_ci=matched_ci,factor_balance=balance,decision=decision)
print('Saved research tables to',OUT)
print('Final holdout and prospective ledger remain untouched.')